In [ ]:
from nba_api.stats.endpoints import leaguegamefinder, teamdashboardbygeneralsplits, boxscoretraditionalv2
import pandas as pd
import time
from requests.exceptions import ReadTimeout, ConnectionError

# Helper function to calculate possessions
def calculate_possessions(fga, fta, orb, tov, opp_fga, opp_fta, opp_orb, opp_tov):
    possessions = 0.5 * ((fga + 0.4 * fta - orb + tov) + (opp_fga + 0.4 * opp_fta - opp_orb + opp_tov))
    return possessions

# Step 1: Get all games in the 2023-24 season
def get_all_games(season='2023-24'):
    print("Fetching games...")
    gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable=season)
    games = gamefinder.get_data_frames()[0]
    print(f"Total games found: {len(games)}")
    return games

# Step 2: Get detailed boxscore data for each game and calculate possessions
def get_possessions_and_efficiencies(games):
    possession_data = []
    for game_id in games['GAME_ID']:
        try:
            for attempt in range(3):  # 尝试最多 3 次
                try:
                    boxscore = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id, timeout=60)
                    break  # 请求成功，退出重试循环
                except (ReadTimeout, ConnectionError) as e:
                    print(f"Error fetching game {game_id}: {e}. Retrying... (Attempt {attempt + 1})")
                    time.sleep(5)  # 等待 5 秒后重试
            else:
                print(f"Failed to fetch data for game {game_id} after 3 attempts.")
                continue

            game_data = boxscore.get_data_frames()[1]  # Team stats
            team_stats = game_data[['TEAM_ID', 'FGA', 'FTA', 'OREB', 'TO', 'PTS']]
            
            # Split into two teams
            team1 = team_stats.iloc[0]
            team2 = team_stats.iloc[1]

            possessions = calculate_possessions(
                team1['FGA'], team1['FTA'], team1['OREB'], team1['TO'],
                team2['FGA'], team2['FTA'], team2['OREB'], team2['TO']
            )
            possession_data.append({
                'GAME_ID': game_id,
                'TEAM1_ID': team1['TEAM_ID'],
                'TEAM1_PTS': team1['PTS'],
                'TEAM2_ID': team2['TEAM_ID'],
                'TEAM2_PTS': team2['PTS'],
                'POSSESSIONS': possessions
            })
        except Exception as e:
            print(f"Error processing game {game_id}: {e}")
    return pd.DataFrame(possession_data)

# Step 3: Process games in batches
def process_games_in_batches(games, batch_size=100):
    total_games = len(games)
    all_possession_data = []
    for start_idx in range(0, total_games, batch_size):
        end_idx = min(start_idx + batch_size, total_games)
        batch_games = games.iloc[start_idx:end_idx]
        print(f"Processing games {start_idx + 1} to {end_idx}...")
        batch_data = get_possessions_and_efficiencies(batch_games)
        all_possession_data.append(batch_data)
        time.sleep(10)  # 每批次等待 10 秒，避免被服务器限流
    return pd.concat(all_possession_data, ignore_index=True)

# Step 4: Get offensive and defensive ratings for each team
def get_team_efficiencies(season='2023-24'):
    print("Fetching team efficiencies...")
    team_dashboard = teamdashboardbygeneralsplits.TeamDashboardByGeneralSplits(season=season)
    team_data = team_dashboard.get_data_frames()[0]
    team_efficiencies = team_data[['TEAM_ID', 'OFF_RATING', 'DEF_RATING']]
    print("Team efficiencies fetched.")
    return team_efficiencies

# Main execution
if __name__ == "__main__":
    games = get_all_games()
    possession_data = process_games_in_batches(games)
    team_efficiencies = get_team_efficiencies()

    # Further analysis or regression can be performed here


Fetching games...
Total games found: 4383
Processing games 1 to 100...
Error fetching game 0042300304: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying... (Attempt 1)
Error fetching game 0042300217: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying... (Attempt 1)
Error fetching game 0042300235: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying... (Attempt 1)
Error fetching game 0042300235: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60). Retrying... (Attempt 2)
Error fetching game 0042300235: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60). Retrying... (Attempt 3)
Failed to fetch data for game 0042300235 after 3 attempts.
Error fetching game 0042300204: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60). Retrying... 